# 🎯 Gemma 4 Fine-Tuning & Inference in Google Colab (T4 GPU)

This companion notebook walks you step-by-step through setting up, testing, and fine-tuning Google's **Gemma 4 (Edge 2B)** model using standard **Hugging Face Transformers, PEFT, and TRL** on a free **NVIDIA T4 GPU** inside Google Colab.

### 🧠 What is Gemma 4 E2B?
The **Gemma 4 E2B** (Edge 2B) is part of Google's lightweight open-weights model family, engineered specifically for edge and mobile environments. It supports native multimodal inputs and fits comfortably inside memory-constrained devices. It comes in two primary forms:
1. **`google/gemma-4-E2B` (Base Model)**: Pre-trained on a massive corpus. It excels at autocompleting text but doesn't understand conversational instruction formats.
2. **`google/gemma-4-E2B-it` (Instruction-Tuned Model)**: Fine-tuned by Google to follow conversational formatting, reply to user prompts, and act as an interactive assistant.

### 🔓 Closed API-only Models vs. 🌌 Open-Weights Models
When developing LLM-powered applications, developers choose between two paradigms:
- **Closed API-only Models (e.g., Gemini API, GPT-4)**: These are black-box services hosted on third-party servers. They are charged on a strict per-token basis, present substantial data privacy/compliance challenges (since proprietary data must cross trust boundaries), and risk service disruptions or model deprecation. Most importantly, you cannot view, modify, or export the underlying neural weights.
- **Open-Weights Models (e.g., Gemma 4)**: These models publish their complete model architecture and parameter weights, giving you 100% ownership and control. You can run them completely offline, host them on your own serverless infrastructure, ensure total data privacy, and modify the internal weight tensors via fine-tuning to perform specialized domain tasks with flawless reliability.

### 🎭 Model Behavioral Comparison Matrix
This matrix displays the exact behavior expected from each model variant given the **exact same prompt**:

| Model Variant | Expected Behavior | Example Output | Linguistic/Statistical Explanation |
| :--- | :--- | :--- | :--- |
| **Instruction-Tuned (IT)**<br>`google/gemma-4-E2B-it` | **Task Aligned Helper**<br>Understands conversational framing, targets system prompts, and responds succinctly. | `Positive` | It has been aligned using **Supervised Fine-Tuning (SFT)** and RLHF to identify conversational markers (e.g. user-assistant templates) and output task compliance. |
| **Base Model**<br>`google/gemma-4-E2B` | **Raw Statistical Autocomplete**<br>Ignores the command and treats the prompt as the start of a document, continuing to list reviews or templates. | `Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'`<br>`Classify the sentiment: 'Bad battery.'`<br>`Classify the sentiment: 'Excellent stay.'` | It acts as a **high-entropy token completer**. It only knows how to continue the statistical pattern of the input text, not how to follow an instruction. |
| **Fine-Tuned Model**<br>`Base Model + Sentiment Adapter` | **Domain Aligned Specialist**<br>Constrains its high-entropy autocomplete states to output *only* your target sentiment label. | `Positive` | QLoRA backpropagation updated the adapter's linear projection layers, steering attention heads to emit only the targeted classification tokens when prompted with the instruction template. |

### 🧬 Parameter-Efficient Fine-Tuning (PEFT) & LoRA
Fine-tuning a modern LLM traditionally required updating every single one of its millions or billions of weights. For a 2-billion parameter model like Gemma 4, full-parameter training is incredibly expensive and demands massive GPU clusters to fit the model weights, gradients, and optimizer states in memory.

**Low-Rank Adaptation (LoRA)** is a highly effective Parameter-Efficient Fine-Tuning (PEFT) method that resolves this. During training, LoRA freezes the original pre-trained weight matrices ($W_0 \in \mathbb{R}^{d \times k}$) and updates only a highly compressed, low-rank representation of the changes.

Instead of directly learning a full weight update matrix $\Delta W \in \mathbb{R}^{d \times k}$, LoRA decomposes it into the product of two extremely skinny, low-rank matrices $A \in \mathbb{R}^{d \times r}$ and $B \in \mathbb{R}^{r \times k}$:
$$\Delta W = A \times B$$
where $r$ is the **rank** (e.g., $r = 8$ or $r = 16$), representing a fraction of the original dimensions ($r \ll \min(d, k)$).

- **Conceptual Example:** Consider a single projection layer where input and output dimensions $d = 2048$ and $k = 2048$.
  - *Full Parameter Fine-Tuning* requires updating and saving $\Delta W$, which has $2048 \times 2048 = 4,194,304$ trainable parameters.
  - *LoRA Fine-Tuning (with Rank $r=16$)* trains matrix $A$ ($2048 \times 16$) and matrix $B$ ($16 \times 2048$), totaling $(2048 \times 16) + (16 \times 2048) = 65,536$ parameters.
  - This represents an outstanding **98.44% reduction in trainable parameters** for that layer! Across the entire model, LoRA reduces trainable parameters by **99%+**, making fine-tuning exceptionally fast and memory-efficient.

### 💾 4-bit Quantization & NormalFloat4 (NF4)
Even with LoRA, loading a 2B parameter model in standard 16-bit precision requires over **4 Gigabytes of VRAM** just to hold the model in GPU memory, which leaves little to no headroom for the batch data, activation maps, and optimization overhead on standard GPUs.

**QLoRA (Quantized Low-Rank Adaptation)** introduces a major breakthrough by compressing the frozen base model down to 4-bit precision using a specialized format called **NormalFloat 4 (NF4)**:
- **NormalFloat 4 (NF4)**: Standard integer quantization (like INT4) uses equally spaced intervals, which performs poorly on neural network weights because weight matrices almost always follow a zero-centered normal distribution. NF4 is an information-theoretically optimal quantile quantization format. It constructs quantization bins such that each of the 16 available 4-bit symbols has an equal expected probability of occurrence, maximizing information density and keeping model accuracy remarkably high despite 4-bit compression.
- **VRAM Footprint Reduction**: Quantizing Gemma 4 E2B to NF4 reduces the VRAM footprint from **4GB down to just ~1.5GB VRAM**. This frees up over **14GB of VRAM** on a standard free-tier NVIDIA T4 GPU (which has 16GB total), leaving massive headroom for high-throughput training batches, gradient storage, and paged optimizers (`paged_adamw_8bit`).

This notebook is **100% self-contained** and can be run end-to-end on Colab's free tier!

---


In [ ]:
# Install required packages for training, quantization, and GCS integrations
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets google-cloud-storage


## 🔑 Step 1: Hugging Face Authentication

Because Gemma 4 is a **gated model**, you must accept the license terms on the Hugging Face model cards before running this notebook:
- [Gemma 4 E2B Base](https://huggingface.co/google/gemma-4-E2B)
- [Gemma 4 E2B IT](https://huggingface.co/google/gemma-4-E2B-it)

Once accepted, grab your **Hugging Face Token** and enter it below. The cell is designed to automatically check for Colab's built-in **Secrets** manager (recommended) or fall back to an interactive login block.

> [!IMPORTANT]
> **🔑 Sharing and Pushing to HF Hub:** If you plan to upload your fine-tuned model weights to the **Hugging Face Hub** at the end of this notebook, make sure to generate and use a Hugging Face Token with **Write** permissions (from your [Hugging Face Settings -> Access Tokens](https://huggingface.co/settings/tokens)). A read-only token can download the models but will fail during push operations.


In [ ]:
import os
try:
    from google.colab import userdata
    # If you have added HF_TOKEN as a Colab Secret (the left key-icon menu), load it directly
    hf_token = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = hf_token
    print("✅ HF_TOKEN detected and loaded from Colab Secrets!")
except Exception:
    print("ℹ️ HF_TOKEN not found in Colab Secrets. Falling back to interactive login...")
    from huggingface_hub import notebook_login
    notebook_login()


## 💬 Step 2: Testing the Instruction-Tuned (IT) Model

We will load `google/gemma-4-E2B-it` in **4-bit quantization** to demonstrate how an aligned conversational model behaves. It will follow your prompt's system instructions and output a clean, formatted sentiment label.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

it_model_id = "google/gemma-4-E2B-it"

# 1. Define 4-bit Quantization (QLoRA config)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # Use float16 for T4 compatibility
    bnb_4bit_use_double_quant=True
)

# 2. Load the Model & Tokenizer
print(f"⏳ Loading instruction-tuned model '{it_model_id}' in 4-bit...")
it_model = AutoModelForCausalLM.from_pretrained(
    it_model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Unwrap Gemma4ClippableLinear modules if present to prevent PEFT/LoRA errors
for name, module in list(it_model.named_modules()):
    if module.__class__.__name__ == 'Gemma4ClippableLinear':
        parts = name.split('.')
        parent = it_model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], module.linear)

it_tokenizer = AutoTokenizer.from_pretrained(it_model_id)
print("✅ Instruction-Tuned model loaded successfully!")

# 3. Test prompt using standard Chat Template
messages = [
    {"role": "user", "content": "Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'"}
]

formatted_prompt = it_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = it_tokenizer(formatted_prompt, return_tensors="pt").to(it_model.device)

print(f"\n[Formatted Prompt for Model]:\n{formatted_prompt}\n")

with torch.no_grad():
    outputs = it_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=it_tokenizer.eos_token_id
    )

prompt_len = inputs.input_ids.shape[1]
generated_ids = outputs[0][prompt_len:]
response = it_tokenizer.decode(generated_ids, skip_special_tokens=True)

print("\n============================================================")
print("🌟 IT MODEL INFERENCE RESULTS & BEHAVIORAL ANALYSIS")
print("============================================================")
print(f"Original Prompt: \"Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'\"")
print(f"Raw Generated Text:\n{response.strip()}")
print("-"*60)
print("💡 ANALYST EXPLANATION:")
print("The Instruction-Tuned (IT) model understands turn-taking syntax because")
print("it has undergone SFT and RLHF alignment. It recognizes user queries inside")
print("conversational templates and behaves as an assistant, targeting your")
print("instruction directly and returning the classification label or structured answer.")
print("============================================================")


### 🎮 Try Your Own Prompts on the IT Model!
Before we clear the model from memory to avoid running out of RAM, use the interactive form-field below to test other prompts of your choice.


In [ ]:
#@title 💬 Interactive IT Model Playground { run: "auto" }
#@markdown Type your own prompt below to test how the aligned instruction-tuned model responds.

custom_prompt = "Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'" #@param {type:"string"}

# Ensure model is still in memory
if 'it_model' in globals() and 'it_tokenizer' in globals():
    messages = [{"role": "user", "content": custom_prompt}]
    formatted_prompt = it_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = it_tokenizer(formatted_prompt, return_tensors="pt").to(it_model.device)
    
    with torch.no_grad():
        outputs = it_model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=it_tokenizer.eos_token_id
        )
    
    prompt_len = inputs.input_ids.shape[1]
    response = it_tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()
    
    print("="*60)
    print("🎭 IT MODEL RESPONSE:")
    print("="*60)
    print(response)
    print("="*60)
else:
    print("⚠️ Error: The Instruction-Tuned model has already been deleted from VRAM. Please re-run the loading cell above.")


### 🧹 Freeing VRAM Memory
Because T4 GPUs are limited, we must delete the loaded IT model and clear PyTorch's cache before loading the Base model to prevent out-of-memory crashes.


In [ ]:
import sys
import gc
import torch

# 1. Clear model, tokenizer, and input references from the namespace
for var in ['it_model', 'it_tokenizer', 'inputs', 'outputs', 'formatted_prompt', 'custom_prompt', 'response']:
    if var in globals():
        del globals()[var]

# 2. Clear traceback references to release memory held by errors
sys.last_traceback = None
sys.last_value = None
sys.last_type = None

# 3. Flush IPython's command history / Out dict cache
try:
    ipython = get_ipython()
    if ipython is not None:
        ipython.user_ns.pop('_', None)
        ipython.user_ns.pop('__', None)
        ipython.user_ns.pop('___', None)
        for key in list(ipython.user_ns.keys()):
            if key.startswith('_') and key[1:].isdigit():
                ipython.user_ns.pop(key, None)
        if 'Out' in ipython.user_ns:
            ipython.user_ns['Out'].clear()
except NameError:
    pass

# 4. Trigger garbage collector & empty PyTorch CUDA cache
gc.collect()
torch.cuda.empty_cache()
print("🧹 VRAM memory cleared! 100% reference cache flushed successfully.")


## 📝 Step 3: Testing the Base Model (Contrastive Autocomplete)

Now, we will load the pre-trained base model `google/gemma-4-E2B` and run the **exact same prompt**.

Observe the output: instead of answering with a classification label, the base model will continue autocompleting reviews or creating similar templates. This is normal base model behavior because it lacks conversational instruction-alignment.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

base_model_id = "google/gemma-4-E2B"

# Ensure 4-bit quantization config is defined (e.g., if Step 2 was skipped during a reload)
if 'bnb_config' not in globals():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

print(f"⏳ Loading base model '{base_model_id}' in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Unwrap Gemma4ClippableLinear modules if present to prevent PEFT/LoRA errors
for name, module in list(base_model.named_modules()):
    if module.__class__.__name__ == 'Gemma4ClippableLinear':
        parts = name.split('.')
        parent = base_model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], module.linear)

base_tokenizer = AutoTokenizer.from_pretrained(base_model_id)
print("✅ Base model loaded successfully!")

# Run the exact same prompt
prompt = "Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'"
inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        pad_token_id=base_tokenizer.eos_token_id
    )

response = base_tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n============================================================")
print("📝 BASE MODEL INFERENCE RESULTS & BEHAVIORAL ANALYSIS")
print("============================================================")
print(f"Original Prompt: \"{prompt}\"")
print(f"Raw Generated Text:\n{response}")
print("-"*60)
print("⚠️ ANALYST EXPLANATION:")
print("Observe how the Base Model completely ignored the command 'Classify the sentiment'!")
print("Instead of outputting 'Positive', it autocompleted our text. It likely added")
print("more hypothetical reviews (e.g. 'Classify the sentiment: Bad battery...', etc.).")
print("This high-entropy document completion is the standard, expected behavior")
print("of an unaligned base model, which acts as a pure statistical text completer.")
print("============================================================")


### 🎮 Try Your Own Prompts on the Base Model!
Use the interactive form below to test other custom prompts on the Base Model. Notice how it behaves as a pure autocomplete engine, continuing whatever pattern you start.


In [ ]:
#@title 📝 Interactive Base Model Playground { run: "auto" }
#@markdown Type your own text prompt below to observe how the unaligned base model autocompletes/continues it.

custom_prompt = "Classify the sentiment: 'I had an absolutely wonderful experience!'" #@param {type:"string"}

if 'base_model' in globals() and 'base_tokenizer' in globals():
    inputs = base_tokenizer(custom_prompt, return_tensors="pt").to(base_model.device)
    
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            pad_token_id=base_tokenizer.eos_token_id
        )
    
    response = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print("="*60)
    print("📝 BASE MODEL AUTOCOMPLETE RESPONSE:")
    print("="*60)
    print(response)
    print("="*60)
else:
    print("⚠️ Error: The Base model has not been loaded yet, or was removed. Please run the Base model loading cell above.")


### 🧹 Freeing VRAM Memory Before Training
To avoid running out of memory during fine-tuning, we must delete our Step 3 inference base model and clear PyTorch's cache. This ensures the GPU has maximum free VRAM for gradient updates and optimizer states during training.


In [ ]:
import sys
import gc
import torch

# 1. Clear model, tokenizer, and input references from the namespace
for var in ['base_model', 'base_tokenizer', 'inputs', 'outputs', 'custom_prompt', 'response']:
    if var in globals():
        del globals()[var]

# 2. Clear traceback references to release memory held by errors
sys.last_traceback = None
sys.last_value = None
sys.last_type = None

# 3. Flush IPython's command history / Out dict cache
try:
    ipython = get_ipython()
    if ipython is not None:
        ipython.user_ns.pop('_', None)
        ipython.user_ns.pop('__', None)
        ipython.user_ns.pop('___', None)
        for key in list(ipython.user_ns.keys()):
            if key.startswith('_') and key[1:].isdigit():
                ipython.user_ns.pop(key, None)
        if 'Out' in ipython.user_ns:
            ipython.user_ns['Out'].clear()
except NameError:
    pass

# 4. Trigger garbage collector & empty PyTorch CUDA cache
gc.collect()
torch.cuda.empty_cache()
print("🧹 VRAM memory cleared and ready for fine-tuning!")


## 📊 Step 4: Programmatic & Agentic Dataset Synthesis

To prepare your Gemma 4 model for training, you have two major pathways for dataset preparation:

### 🧱 Pathway A: Local Programmatic Script (Included Below)
To make this notebook **100% self-contained**, we include a local python cell that programmatically compiles **500 sentiment analysis samples** covering varied linguistic nuances (such as double negations, slang, sarcasm, and emoji emphasis) and saves them locally to `sentiment_dataset.jsonl` matching Gemma's conversational training schema:
```json
{
  "messages": [
    {"role": "user", "content": "Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'"},
    {"role": "model", "content": "Positive"}
  ]
}
```

---

### 🚀 Pathway B: Agentic Synthesis with Google Antigravity (AGY)
While programmatic template generation is useful for a quick demo, it has highly limited vocabulary diversity. For production-grade fine-tuning, you need high-fidelity synthetic data. By using the **AGY CLI (`agy`)** or **Antigravity 2.0 Desktop app**, you can command the AI to act as a **Producer Archetype** to generate thousands of unique, natural, and complex samples with structured schema constraints.

#### 💡 Other Domain Ideas & Precise AGY Prompts
Here are three other high-value domain-specific use cases you can fine-tune Gemma 4 on, along with the exact prompts you can copy and paste directly into **AGY** to generate your custom datasets:

##### 1️⃣ Multilingual Support Ticket Router
* **Concept:** Fine-tune Gemma to act as an automated ticket triage gateway, classifying raw customer messages (English, Spanish, French, German, Japanese, Hindi) into issues (`billing`, `tech_support`, `account_security`, `refund_request`) and priorities (`critical`, `high`, `medium`, `low`).
* **AGY Prompt:**
  ```
  Role: Act as a data generation engineer (Producer Archetype).
  Task: Create a synthetic fine-tuning dataset of 1000 customer support triage samples.
  Constraints:
  - The input support ticket should be in random languages (English, Spanish, French, German, Japanese, Hindi).
  - The model must output a JSON block indicating category and priority.
  - Format: Save as a JSON Lines (.jsonl) file in my current workspace named `triage_dataset.jsonl`.
  - Schema: Every line must be a single JSON object matching this structure exactly:
    {"messages": [{"role": "user", "content": "Triage this support ticket: '<TICKET_TEXT>'"}, {"role": "model", "content": "{\"category\": \"<CAT>\", \"priority\": \"<PRIORITY>\"}"}]}
  ```

##### 2️⃣ Text-to-API Payload Copilot (Function Caller)
* **Concept:** Fine-tune Gemma to translate unstructured natural language request intents into structured REST API JSON payloads for backend microservices.
* **AGY Prompt:**
  ```
  Role: Act as a data generation engineer (Producer Archetype).
  Task: Create a synthetic fine-tuning dataset of 1000 Text-to-API translation samples.
  Constraints:
  - The user prompt should be a natural language request to perform an action (e.g., booking, profile updates).
  - The model response should be a formatted, structured REST API payload.
  - Format: Save as a JSON Lines (.jsonl) file in my current workspace named `api_dataset.jsonl`.
  - Schema: Every line must be a single JSON object matching this structure exactly:
    {"messages": [{"role": "user", "content": "Translate to API payload: '<REQUEST_TEXT>'"}, {"role": "model", "content": "{\"action\": \"<ACTION>\", \"params\": {<PARAMETERS>}}"}]}
  ```

##### 3️⃣ Enterprise PII Redactor
* **Concept:** Fine-tune Gemma to redact/mask sensitive customer information (emails, names, SSNs, credit card numbers, phone numbers) before logging chat logs.
* **AGY Prompt:**
  ```
  Role: Act as a data generation engineer (Producer Archetype).
  Task: Create a synthetic fine-tuning dataset of 1000 PII anonymization samples.
  Constraints:
  - The input should be a conversational transcript containing names, credit card numbers, email addresses, phone numbers, or SSNs.
  - The model response must be the exact same transcript, but with all PII replaced by standardized tags like `[REDACTED_NAME]`, `[REDACTED_EMAIL]`, `[REDACTED_PHONE]`, `[REDACTED_CARD]`.
  - Format: Save as a JSON Lines (.jsonl) file in my current workspace named `pii_dataset.jsonl`.
  - Schema: Every line must be a single JSON object matching this structure exactly:
    {"messages": [{"role": "user", "content": "Redact PII from this transcript: '<TRANSCRIPT>'"}, {"role": "model", "content": "<REDACTED_TRANSCRIPT>"}]}
  ```

Let's execute Pathway A to programmatically synthesize our sentiment dataset and run the training process.


In [ ]:
import json
import random

# Define lexicon databases for synthesis
domains = ["product", "movie", "app", "restaurant", "stay"]
templates = {
    "pos": [
        "This {domain} is incredible and highly recommended.",
        "Absolutely loved my stay, everything was spotless!",
        "It works like a charm, worth every single dollar.",
        "I don't dislike this {domain} at all, it's amazing.",
        "This is easily the GOAT! Extremely top-tier 🔥",
        "Although the setup was steep, it is spectacular!"
    ],
    "neg": [
        "Worst {domain} I have ever spent money on.",
        "Oh great, another bug-filled disaster... exactly what I needed.",
        "It broke down after only three days of light usage.",
        "Avoid at all costs. Absolute waste of time.",
        "Cheap material, clunky interface, and sluggish support.",
        "I really wanted to support this, but it is a complete trainwreck."
    ]
}

synthetic_data = []
for i in range(500):
    sentiment = random.choice(["Positive", "Negative"])
    tpl_group = "pos" if sentiment == "Positive" else "neg"
    raw_review = random.choice(templates[tpl_group]).format(domain=random.choice(domains))
    
    # Randomize instruction wrapping
    inst = f"Classify the sentiment: '{raw_review}'"
    
    synthetic_data.append({
        "messages": [
            {"role": "user", "content": inst},
            {"role": "model", "content": sentiment}
        ]
    })

# Save locally to file
dataset_path = "sentiment_dataset.jsonl"
with open(dataset_path, "w", encoding="utf-8") as f:
    for sample in synthetic_data:
        f.write(json.dumps(sample) + "\n")

print(f"✅ Successfully synthesized 500 samples in: '{dataset_path}'")


## 🏋️ Step 5: Fine-Tuning Gemma Base with QLoRA

Now we will load our dataset into the Hugging Face `SFTTrainer` and execute our QLoRA training run on the T4 GPU.

> [!IMPORTANT]
> **💥 CRITICAL: Google Colab T4 Memory Management & Session Restart Protocol**
> Because we loaded and played with both the **Instruction-Tuned** and **Base** models in Step 2 and Step 3, Python's garbage collector and Google Colab's interactive forms may have lingering tensor references in memory. Even minor VRAM leaks of 1-2 GB will cause an `OutOfMemoryError` during training because `prepare_model_for_kbit_training` and gradient updates require every megabyte of our 15GB VRAM.
>
> **🔄 To ensure a 100% successful training run with ZERO memory issues:**
> 1. Go to the top menu and select **Runtime -> Restart session** (or use shortcut `Ctrl + M` then `.`).
>    * *Note: Restarting the session clears the Python state and frees 100% of VRAM, but **keeps** all pip installations and files intact!*
> 2. Once the session is restarted, skip the loading cells in Step 2 & 3, and run **Step 1** (HF Authentication) to load your credentials.
> 3. Proceed directly to **Step 5** below to begin fine-tuning!

### How the adapters are applied:
1. We freeze the loaded 4-bit base model `google/gemma-4-E2B` to prevent its original parameters from shifting.
2. We define a `LoraConfig` that targets only the projection modules. This inserts low-rank matrices into the network, isolating trainable params.
3. We trigger `SFTTrainer` which runs sequence batching, computes loss, and updates our adapter matrices.


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

# 1. Load the generated dataset
dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"Dataset Loaded. Record Count: {len(dataset)}")

# 2. Reload Base Model & Tokenizer Fresh for Fine-Tuning
base_model_id = "google/gemma-4-E2B"
print(f"⏳ Loading fresh base model '{base_model_id}' in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # Use float16 for T4 compatibility
    bnb_4bit_use_double_quant=True
)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16, # Load unquantized weights in float16 to prevent bfloat16 mixed-type GradScaler errors on T4
    device_map="auto"
)

# Unwrap Gemma4ClippableLinear modules if present to prevent PEFT/LoRA errors
for name, module in list(base_model.named_modules()):
    if module.__class__.__name__ == 'Gemma4ClippableLinear':
        parts = name.split('.')
        parent = base_model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], module.linear)

base_tokenizer = AutoTokenizer.from_pretrained(base_model_id)
print("✅ Fresh Base model loaded successfully!")

# 3. Custom memory-efficient prepare_model_for_kbit_training
def custom_prepare_model_for_kbit_training(model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs=None):
    for name, param in model.named_parameters():
        param.requires_grad = False
    
    # Only upcast tiny normalization layers (norm, ln) to float32 for stability.
    # We skip upcasting the massive 2.8B parameter embedding & language model head layers,
    # saving ~8.75 GiB of VRAM from being allocated!
    for name, param in model.named_parameters():
        if ("norm" in name or "ln" in name) and param.__class__.__name__ != "Params4bit":
            if param.dtype in [torch.float16, torch.bfloat16]:
                param.data = param.data.to(torch.float32)
    
    # Explicitly convert any remaining bfloat16 parameters and buffers to float16 (essential for T4 GPU compatibility)
    for name, param in model.named_parameters():
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)
    for name, buf in model.named_buffers():
        if buf.dtype == torch.bfloat16:
            buf.data = buf.data.to(torch.float16)
    
    if use_gradient_checkpointing:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gradient_checkpointing_kwargs)
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()
        else:
            def make_inputs_require_grad(module, input, output):
                output.requires_grad_(True)
            model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)
    return model

base_model = custom_prepare_model_for_kbit_training(base_model)
base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = "right"

# Set standard Gemma chat template since we are using a conversational messages dataset
base_tokenizer.chat_template = (
    "{{ bos_token }}"
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}"
    "{{ '<start_of_turn>user\\n' + message['content'] + '<end_of_turn>\\n' }}"
    "{% elif message['role'] == 'model' %}"
    "{{ '<start_of_turn>model\\n' + message['content'] + '<end_of_turn>\\n' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<start_of_turn>model\\n' }}"
    "{% endif %}"
)

# 4. Configure LoRA parameters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

# 5. Define modern TRL SFT Training Parameters
output_dir = "./results"
training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=1, # 1 Epoch is enough for this self-contained demo
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2e-4,
    optim="paged_adamw_8bit", # Saves optimizer memory
    fp16=True, # Use FP16 for standard T4 operations
    bf16=False,
    logging_steps=10,
    save_strategy="no",
    max_length=512,
    report_to="none"
)

# 6. Initialize SFTTrainer
trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=base_tokenizer,
    args=training_args
)

# Explicitly align parameter dtypes for mixed-precision training on T4 GPU:
# 1. Trainable weights (LoRA adapters) must remain in float32 for GradScaler gradient accumulation.
# 2. Frozen weights (embeddings, head) must be cast from bfloat16 to float16 to prevent bfloat16 activation leakage.
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        if param.dtype != torch.float32:
            param.data = param.data.to(torch.float32)
    else:
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)
for name, buf in trainer.model.named_buffers():
    if buf.dtype == torch.bfloat16:
        buf.data = buf.data.to(torch.float16)

# 7. Execute Fine-Tuning!
print("⏳ Starting QLoRA fine-tuning...")
trainer.train()
print("✅ Fine-Tuning complete!")

# 8. Save Adapter locally
adapter_dir = "./fine_tuned_gemma_adapter"
trainer.model.save_pretrained(adapter_dir)
base_tokenizer.save_pretrained(adapter_dir)
print(f"💾 Fine-tuned adapter saved locally to '{adapter_dir}'")


## 💾 Step 5.5: Ephemerality Protection – Saving, Exporting, and Restoring Your Fine-Tuned Adapter

Congratulations! You have successfully fine-tuned the raw Base Model into a specialized Sentiment Classifier. However, we have a critical challenge to address: **ephemerality**.

### ⚠️ The Reality of Colab VM Ephemerality
Google Colab operates on an **ephemeral containerized model**. When you start a notebook, you are leased a temporary virtual machine (VM) backed by a cloud GPU. However, this lease is highly volatile and completely non-persistent:
- **Inactivity Timeouts**: If you do not interact with the notebook for a short period, Colab automatically disconnects your session.
- **Browser/Tab Closure**: Closing your browser tab or experiencing a brief internet outage can trigger container termination.
- **Lease Expirations**: Free-tier sessions have hard time limits (usually 12 hours max) after which the VM is forcefully reclaimed.
- **Runtime Restarts**: Changing runtime types, GPU allocations, or experiencing memory crashes instantly reboots the kernel.

**When any of these events occur, the virtual hard drive of your Colab container is instantly and permanently wiped clean.** Any files stored under `/content/` or `./fine_tuned_gemma_adapter` are deleted forever. To protect your investment in training time and compute, you MUST persist your weights to an external, durable storage provider immediately.

### 🌌 The Power of Portable Adapters: 15MB vs. 10GB
Fortunately, because we utilized **QLoRA (Quantized Low-Rank Adaptation)**, we did not alter the original pre-trained base model weights. Instead, we only trained small, auxiliary low-rank matrices. This architectural separation yields a staggering difference in portability:

| Characteristic | Frozen Base Model | QLoRA Sentiment Adapter |
| :--- | :--- | :--- |
| **Parameter Count** | ~2,000,000,000 (2 Billion) | ~13,000,000 (13 Million) |
| **File Size on Disk** | **~10.2 GB** (Gigabytes) | **~15 MB** (Megabytes) |
| **Transfer/Save Time** | Extremely slow (~5-15 mins over network) | Instantaneous (~1-2 seconds) |
| **Storage Cost** | High (Requires premium cloud space) | Virtually Free |
| **Storage Destination**| Hugging Face Cache (Read-only) | GCS, Hugging Face, GDrive, Local |

### 🗺️ The Fine-Tuning and Saving Lifecycle
The following engineering diagram illustrates how our frozen base model combines with trainable adapter layers during training, and how we can persist these tiny weights permanently and restore them in subsequent sessions in **2 seconds flat**—completely bypassing retraining!

| Model Variant | Expected Behavior | Example Output |
| :--- | :--- | :--- |
| **Instruction-Tuned (IT)** | **Task Aligned Helper** | `Positive` |

![Gemma 4 Fine-Tuning and Saving Workflow](gemma_finetuning_workflow.png)

### ⚡ Dynamic 2-Second Restoration in Fresh Sessions
This incredible difference in size is what makes open-weights edge models so powerful in production. Because the base model `google/gemma-4-E2B` is standardized and publicly hosted on Hugging Face, you do not need to download or back up its massive 10GB weights. You only need to save your tiny **15MB adapter folder**.

In any fresh session in the future, you can simply load the raw base model dynamically in 4-bit, and then download and overlay your custom 15MB adapter on top of it. The PEFT library merges these weights at runtime in **less than 2 seconds**, giving you a fully restored, domain-aligned specialist model without ever having to re-run the fine-tuning process!

Because our adapter is so lightweight, we can easily back it up using multiple methods. Below, you will find four optional ways to save your adapter (Part A), and a unified restoration block to reload them on a fresh runtime (Part B).


### 📂 Part A: Save & Export Your Adapter (Choose One)
Choose **one** of the four options below to save your fine-tuned weights permanently. You do not need to run all of them.


#### 📁 Option 1: Backup directly to your personal Google Drive
This is the easiest and most convenient option. It mounts your Google Drive as a local folder and copies the adapter directory directly to your `MyDrive` space.


In [ ]:
#@title 📁 Option 1: Save Adapter to Google Drive { run: "none" }
import shutil
import os

try:
    from google.colab import drive
    print("⏳ Mounting Google Drive...")
    drive.mount('/content/drive')
    
    gdrive_adapter_path = "/content/drive/MyDrive/fine_tuned_gemma_adapter"
    
    print(f"⏳ Copying adapter weights from local session to Google Drive at '{gdrive_adapter_path}'...")
    if os.path.exists(gdrive_adapter_path):
        shutil.rmtree(gdrive_adapter_path)
    shutil.copytree("./fine_tuned_gemma_adapter", gdrive_adapter_path)
    print("🎉 SUCCESS! Your fine-tuned adapter has been saved permanently to your Google Drive.")
    print("💡 You can now safely close this Colab runtime or restart the session.")
except Exception as e:
    print(f"❌ Failed to save to Google Drive (make sure to grant permissions): {e}")


#### 🤗 Option 2: Share and Push to the Hugging Face Hub (Highly Recommended)
This is the industry-standard way to save model weights. Pushing your adapter to Hugging Face creates a repository (which can be **public** or **private**) from which you or anyone else can load the weights in a single line of code! (Make sure you logged in with a **WRITE** token in Step 1).


In [ ]:
#@title 🤗 Option 2: Push Adapter to Hugging Face Hub { run: "none" }
#@markdown Enter your Hugging Face username and desired repository name below.
HF_USERNAME = "your_hf_username" #@param {type:"string"}
HF_REPO_NAME = "gemma-4-sentiment-adapter" #@param {type:"string"}
MAKE_PRIVATE = True #@param {type:"boolean"}

hub_model_id = f"{HF_USERNAME}/{HF_REPO_NAME}"

if HF_USERNAME == "your_hf_username":
    print("⚠️ Please enter your real Hugging Face username in the form-field above!")
else:
    print(f"⏳ Pushing adapter weights and tokenizer to Hugging Face Hub: {hub_model_id}...")
    try:
        # Push the PEFT adapter model and base tokenizer
        trainer.model.push_to_hub(hub_model_id, private=MAKE_PRIVATE)
        base_tokenizer.push_to_hub(hub_model_id, private=MAKE_PRIVATE)
        print(f"\n🎉 Successfully pushed to HF Hub! View it at: https://huggingface.co/{hub_model_id}")
        print("\n💡 Others can load and use your adapter directly using:")
        print(f"   PeftModel.from_pretrained(base_model, \"{hub_model_id}\")")
    except Exception as e:
        print(f"\n❌ Push failed: {e}")
        print("💡 Did you use a token with WRITE permissions during login in Step 1?")


#### ☁️ Option 3: Export to Google Cloud Storage (GCS) (Production & Cloud Run Integration)
This option uploads your adapter directly to a Google Cloud Storage bucket. This is ideal if you want to integrate this adapter with our production deployment workflow, where your Cloud Run serving service downloads the weights dynamically during startup.


In [ ]:
#@title ☁️ Option 3: Upload Adapter to GCS { run: "none" }
#@markdown Authenticates your GCP account, creates a bucket, and uploads the folder.
GCP_PROJECT_ID = "YOUR_GCP_PROJECT_ID" #@param {type:"string"}
GCS_BUCKET_NAME = "your-gemma-gcp-bucket" #@param {type:"string"}
GCS_DEST_FOLDER = "gemma-4-adapters/" #@param {type:"string"}

if GCP_PROJECT_ID == "YOUR_GCP_PROJECT_ID":
    print("⚠️ Please enter your active GCP Project ID in the form-field above!")
else:
    try:
        from google.colab import auth
        print("⏳ Authenticating Google Cloud account within Colab...")
        auth.authenticate_user()
        
        # Set active gcloud project
        !gcloud config set project {GCP_PROJECT_ID}
        
        # Create GCS bucket if not exists
        print(f"⏳ Checking/Creating GCS bucket: gs://{GCS_BUCKET_NAME}...")
        !gcloud storage buckets create gs://{GCS_BUCKET_NAME} --location=us-central1 || true
        
        # Upload local fine-tuned adapter directory to GCS
        print(f"⏳ Uploading local adapter files to gs://{GCS_BUCKET_NAME}/{GCS_DEST_FOLDER}...")
        !gcloud storage cp -r ./fine_tuned_gemma_adapter gs://{GCS_BUCKET_NAME}/{GCS_DEST_FOLDER}
        print(f"\n🎉 Successfully uploaded fine-tuned adapter weights to: gs://{GCS_BUCKET_NAME}/{GCS_DEST_FOLDER}")
    except Exception as e:
        print(f"❌ GCS upload failed: {e}")


#### 📥 Option 4: Download directly to your local computer (No accounts required)
If you want to keep a local copy of your fine-tuned weights without setting up cloud buckets or online registries, this block will compress your adapter folder into a `.zip` archive and trigger a direct browser download.

**💡 Why is the archive ~95 MB?**
Many beginners are surprised that a LoRA adapter is 95 MB. Here is the exact breakdown:
1. **LoRA Adapter Weights** (`adapter_model.safetensors` - **~62.7 MB**): Hugging Face saves trainable parameters in standard Single-Precision Float (`float32`, 4 bytes/parameter) by default to prevent precision degradation. With ~15 million trainable parameters, this takes exactly `15M * 4 bytes ≈ 60 MB`.
2. **Tokenizer Vocabulary** (`tokenizer.json` - **~32.2 MB**): Gemma 4 uses an ultra-large, modern vocabulary of 256,000 tokens. Saving the tokenizer writes this entire vocabulary index alongside your weights so that the adapter remains fully self-contained.

**Choose your download mode below:**
- **Full Deployment Package** (~95 MB): Includes the adapter weights and the tokenizer. Use this if you plan to upload to the Hugging Face Hub or load it on a system without internet access.
- **Slim Weights-Only Package** (~60 MB): Excludes the tokenizer. Because our reloading cell (Part B) already pulls the base Gemma 4 tokenizer from Hugging Face, you do *not* need to backup the tokenizer to reload your weights locally! This saves bandwidth and disk space.


In [ ]:
#@title 📥 Option 4: Download Adapter as .zip { run: "none" }
DOWNLOAD_MODE = "Full Deployment Package (~95MB)" #@param ["Full Deployment Package (~95MB)", "Slim Weights-Only (~60MB)"]

import os
import zipfile
from google.colab import files

zip_path = "./fine_tuned_gemma_adapter.zip"
adapter_dir = "./fine_tuned_gemma_adapter"

print(f"⏳ Compressing fine-tuned adapter folder using mode: {DOWNLOAD_MODE}...")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_dir in os.walk(adapter_dir):
        for file in files_in_dir:
            file_path = os.path.join(root, file)
            # If slim mode, skip the large tokenizer.json vocabulary file
            if DOWNLOAD_MODE == "Slim Weights-Only (~60MB)" and file == "tokenizer.json":
                print("   Skipping 'tokenizer.json' to keep download slim...")
                continue
            # Maintain relative path inside zip
            arcname = os.path.relpath(file_path, os.path.dirname(adapter_dir))
            zipf.write(file_path, arcname)

print(f"✅ Compression complete! File size: {os.path.getsize(zip_path) / (1024*1024):.2f} MB")
print("⏳ Starting browser download. Please allow popups if prompted...")
files.download(zip_path)


### 🔄 Part B: Restore & Reload Weights (Bypass Retraining!)

If you have already trained your model and saved your adapter using one of the methods above, you **do not need to run Step 5 (the training loop) again!**

When returning to this notebook in a fresh session next time:
1. Run the **Installation cell** at the very top of the notebook to install necessary packages.
2. Run **Step 1** (Hugging Face Authentication) to load your credentials and log in.
3. Run **Step 3** (Testing the Base Model) to load the raw base model `google/gemma-4-E2B` in 4-bit quantization. *(Note: You can completely skip Step 2 to save time and VRAM!)*
4. Run this interactive form cell below. Select your **LOAD_SOURCE**, configure the variables, and click **Run**. This will pull your weights in seconds and instantly configure your model for inference!


In [ ]:
#@title 🔄 Restore Adapter weights { run: "none" }
LOAD_SOURCE = "Google Drive" #@param ["Local Directory", "Google Drive", "Hugging Face Hub", "Browser Upload (.zip)", "Google Cloud Storage (GCS)"]
HF_MODEL_ID = "username/gemma-4-sentiment-adapter" #@param {type:"string"}
GCP_PROJECT_ID = "YOUR_GCP_PROJECT_ID" #@param {type:"string"}
GCS_BUCKET_NAME = "your-gemma-gcp-bucket" #@param {type:"string"}
GCS_ADAPTER_FOLDER = "gemma-4-adapters/" #@param {type:"string"}

import os
import shutil
import zipfile
from peft import PeftModel

adapter_path = "./fine_tuned_gemma_adapter"
upload_successful = True

if LOAD_SOURCE == "Google Drive":
    try:
        from google.colab import drive
        print("⏳ Mounting Google Drive to restore adapter...")
        drive.mount('/content/drive')
        gdrive_path = "/content/drive/MyDrive/fine_tuned_gemma_adapter"
        if os.path.exists(gdrive_path):
            adapter_path = gdrive_path
            print(f"✅ Successfully located adapter in Google Drive: {adapter_path}")
        else:
            print("⚠️ Warning: Adapter folder not found in Google Drive! Make sure you saved it first.")
            upload_successful = False
    except Exception as e:
        print(f"❌ Failed to mount Google Drive: {e}")
        upload_successful = False

elif LOAD_SOURCE == "Browser Upload (.zip)":
    try:
        from google.colab import files
        print("⏳ Please upload your 'fine_tuned_gemma_adapter.zip' file...")
        uploaded = files.upload()
        if uploaded:
            for fn in uploaded.keys():
                if fn.endswith('.zip'):
                    print(f"✅ Uploaded '{fn}'")
                    # Extract zip contents to ./fine_tuned_gemma_adapter
                    if os.path.exists("./fine_tuned_gemma_adapter"):
                        shutil.rmtree("./fine_tuned_gemma_adapter")
                    
                    # Create temporary folder for extraction
                    temp_extract = "./temp_extract_adapter"
                    if os.path.exists(temp_extract):
                        shutil.rmtree(temp_extract)
                    os.makedirs(temp_extract)
                    
                    with zipfile.ZipFile(fn, 'r') as zip_ref:
                        zip_ref.extractall(temp_extract)
                    
                    # Relocate extracted folder correctly if nested
                    extracted_contents = os.listdir(temp_extract)
                    if len(extracted_contents) == 1 and os.path.isdir(os.path.join(temp_extract, extracted_contents[0])):
                        shutil.copytree(os.path.join(temp_extract, extracted_contents[0]), "./fine_tuned_gemma_adapter")
                    else:
                        shutil.copytree(temp_extract, "./fine_tuned_gemma_adapter")
                    
                    shutil.rmtree(temp_extract)
                    print("🎉 Extracted adapter directory to './fine_tuned_gemma_adapter' successfully!")
        else:
            print("❌ Upload cancelled or failed.")
            upload_successful = False
    except Exception as e:
        print(f"❌ Failed to upload or extract zip: {e}")
        upload_successful = False

elif LOAD_SOURCE == "Google Cloud Storage (GCS)":
    if GCP_PROJECT_ID == "YOUR_GCP_PROJECT_ID":
        print("⚠️ Please enter your active GCP Project ID in the form-field above!")
        upload_successful = False
    else:
        try:
            from google.colab import auth
            print("⏳ Authenticating GCP Account...")
            auth.authenticate_user()
            
            # Configure project
            !gcloud config set project {GCP_PROJECT_ID}
            
            # Clean local directory
            if os.path.exists("./fine_tuned_gemma_adapter"):
                shutil.rmtree("./fine_tuned_gemma_adapter")
            os.makedirs("./fine_tuned_gemma_adapter")
            
            # Download adapter weights
            print(f"⏳ Downloading adapter files from GCS: gs://{GCS_BUCKET_NAME}/{GCS_ADAPTER_FOLDER}...")
            !gcloud storage cp -r gs://{GCS_BUCKET_NAME}/{GCS_ADAPTER_FOLDER}* ./fine_tuned_gemma_adapter/
            print("🎉 SUCCESS! Successfully downloaded and restored adapter weights from GCS!")
        except Exception as e:
            print(f"❌ Failed to download from GCS: {e}")
            upload_successful = False

elif LOAD_SOURCE == "Hugging Face Hub":
    adapter_path = HF_MODEL_ID
    print(f"✅ Configured to load adapter from Hugging Face Hub: {adapter_path}")
else:
    print(f"✅ Configured to load from local directory: {adapter_path}")

if upload_successful:
    try:
        print("⏳ Wrapping base model with PEFT adapter...")
        if 'base_model' in globals():
            # Load/re-load the PEFT model wrapper
            if hasattr(base_model, "unload"):
                base_model = base_model.unload() # Reset to base model state
            
            # Load PEFT adapter
            trainer_model = PeftModel.from_pretrained(base_model, adapter_path)
            
            # Ensure 'trainer.model' is globally set so the downstream evaluation cells use it
            if 'trainer' in globals():
                trainer.model = trainer_model
            else:
                # Create a mock trainer class to keep downstream cells running perfectly
                class MockTrainer:
                    pass
                trainer = MockTrainer()
                trainer.model = trainer_model
                
            print("🎉 SUCCESS! Model successfully wrapped with the pre-trained adapter weights.")
            print("💡 You are now ready to run Step 6 (Evaluation) directly!")
        else:
            print("❌ Error: 'base_model' is not loaded in memory. Please run Step 1 and Step 3 first!")
    except Exception as e:
        print(f"❌ Failed to load adapter: {e}")


## 🔬 Step 6: Evaluating the Fine-Tuned Model

Now let's test our newly fine-tuned model (which is the frozen base model merged with our newly trained LoRA adapter).

Observe the output: the base model, which formerly autocompleted random text, has now successfully aligned to behave as a **sentiment classifier**, outputting the clean sentiment label!

### 💡 Under the Hood: What is happening during evaluation?
When we execute `trainer.model.generate(...)`, Hugging Face's `PeftModel` routes inputs through both our frozen base model and our active LoRA adapter layer. The adapter dynamically adjusts the attention weight computations, shifting the token probability distributions so that high-entropy autocompletions are suppressed, and the exact classification labels (`Positive` or `Negative`) emerge with high probability.


In [ ]:
from peft import PeftModel

# Let's run a test prompt directly on our current model state
test_reviews = [
    "Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'",
    "Classify the sentiment: 'Worst service ever, completely broken.'"
]

trainer.model.eval()
trainer.model.config.use_cache = False # Disable KV Cache during PEFT evaluation to prevent PyTorch dimension conflicts on T4

print("\n============================================================")
print("🚀 FINE-TUNED SPECIALIST MODEL INFERENCE RESULTS & ANALYSIS")
print("============================================================")

for pr in test_reviews:
    eval_messages = [{"role": "user", "content": pr}]
    formatted_prompt = base_tokenizer.apply_chat_template(eval_messages, tokenize=False, add_generation_prompt=True)
    inputs = base_tokenizer(formatted_prompt, return_tensors="pt").to(base_model.device)
    inputs.pop("token_type_ids", None) # Remove token_type_ids if present to avoid generation dimension mismatch
    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=base_tokenizer.eos_token_id
        )
        
    prompt_len = inputs.input_ids.shape[1]
    generated_ids = outputs[0][prompt_len:]
    label = base_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    print(f"Prompt entered:  '{pr}'")
    print(f"Model Output:    '{label}'")
    print("-"*60)

print("\n💡 ANALYST EXPLANATION:")
print("This is a dramatic transformation! The raw Base Model, which previously")
print("rambled into high-entropy autocompletion loops, is now behaving as a")
print("perfectly constrained sentiment classification engine. By updating only")
print("a tiny fraction of parameters in the projection layers via QLoRA, we")
print("have steered the model's attention heads to output only our target labels")
print("('Positive' or 'Negative') with zero conversational fluff.")
print("============================================================")


### 🎮 Try Your Own Reviews on the Fine-Tuned Model!
Use the interactive form below to write your own reviews and see how our newly trained sentiment classification specialist classifies them.


In [ ]:
#@title 🚀 Interactive Fine-Tuned Model Playground { run: "auto" }
#@markdown Type your own review prompt below to test your fine-tuned sentiment classifier!

custom_prompt = "Classify the sentiment: 'The food was cold and the waiter was incredibly rude.'" #@param {type:"string"}

# Ensure fine-tuned model is in-memory
if 'trainer' in globals() and 'base_tokenizer' in globals():
    trainer.model.eval()
    trainer.model.config.use_cache = False # Disable KV Cache during PEFT evaluation to prevent PyTorch dimension conflicts on T4
    eval_messages = [{"role": "user", "content": custom_prompt}]
    formatted_prompt = base_tokenizer.apply_chat_template(eval_messages, tokenize=False, add_generation_prompt=True)
    inputs = base_tokenizer(formatted_prompt, return_tensors="pt").to(base_model.device)
    inputs.pop("token_type_ids", None) # Remove token_type_ids if present to avoid generation dimension mismatch
    
    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=base_tokenizer.eos_token_id
        )
        
    prompt_len = inputs.input_ids.shape[1]
    response = base_tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()
    
    print("="*60)
    print("🚀 FINE-TUNED MODEL SPECIALIST RESPONSE:")
    print("="*60)
    print(response)
    print("="*60)
else:
    print("⚠️ Error: The Fine-tuned model or trainer is not in memory. Please complete the QLoRA training steps above.")


## 🚀 Next Steps: Serving Your Fine-Tuned Model in Production

Congratulations! You have completed the entire open-weights fine-tuning pipeline. Let's summarize what we have accomplished:
1. **Compared Behaviors:** Understood why base models differ from instruction-tuned models.
2. **Loaded the base model in 4-bit:** Reduced memory consumption to run comfortably on a standard GPU.
3. **SFT Data Processing:** Loaded and prepared our sentiment reviews using Chat Templates.
4. **Fine-tuned with QLoRA:** Trained tiny, extremely targeted adapters to steer the model's behavior.
5. **Created Ephemerality Backups:** Saved our adapter weights permanently.

### 🌐 Deploying to Google Cloud Run
Now that your adapter weights are safely stored in GCS (Option 3) or Hugging Face (Option 2), you can immediately transition to serving this model in production:
- Check out our local server script **`serve.py`** in this workspace. It sets up an OpenAI-compatible FastAPI endpoints and downloads your adapters dynamically during startup.
- We have provided a complete Docker setup (`Dockerfile.serve` and `service.yaml`) so you can deploy this serving container directly to **Google Cloud Run on Serverless GPUs** in just one command!

Keep up the amazing work in your AI journey!
